# BoW for Review

Ми отримали очищений текст який тепер треба перетворити на числа, векторизувати.  
Для BoW стандартом використання є TfidfVectorizer.  
Він економить пам'ять, автоматично визначає вагу слова, наскільки воно рідкісне та важливе.  
Отримаємо ефективну статистичну матрицю для лінійних моделей.

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import joblib

In [2]:
train_ds = pd.read_csv('train_test_val/train/train_bow.csv')

In [3]:
train_ds.head()

,Rating,Recommended,full_review,clean_text
0,5,1,Excellent I bought this in store on sale and c...,excellent buy store sale completely love
1,5,1,Magnificent shirt!!! I have several of goodhyo...,magnificent shirt goodhyouman shirt compliment...
2,5,1,Cute and versatile Picked this up at the local...,cute versatile pick local retailer color red v...
3,2,0,"Not if you're busty I am a 32dd, 5'4'' and 125...",busty dd lbs way baggy look awful shapeless ...
4,4,1,Neutral blue I tried this on from the sale sec...,neutral blue try sale section usually unex...


In [4]:
tfidf = TfidfVectorizer(max_features=5000)

In [5]:
X_vec = tfidf.fit_transform(train_ds['clean_text'])
X_vec.shape

(10870, 5000)

In [6]:
y_rec = train_ds['Recommended']
y_rec.shape

(10870,)

In [7]:
y_rat = train_ds['Rating']
y_rat.shape

(10870,)

In [8]:
rec_target = y_rec.value_counts()
disb = rec_target.iloc[0] / rec_target.iloc[1]
disb = disb.item()
print(f'Disbalance: {disb:.2f}')

Disbalance: 4.61


In [9]:
rec_target

Recommended
1    8931
0    1939
Name: count, dtype: int64

Маємо дисбаланс класів для рекомендацій.  
Мінорний клас - 0 (не рекомендується)

In [10]:
weight_1 = y_rec.shape[0] / (rec_target.shape[0] * rec_target[1])
weight_1 = weight_1.item()
weight_1

0.6085544731832941

In [11]:
weight_0 = y_rec.shape[0] / (rec_target.shape[0] * rec_target[0])
weight_0 = weight_0.item()
weight_0

2.802991232594121

In [12]:
weights = {0: weight_0, 1: weight_1}

### Logistic Regression

In [13]:
bow_logreg = LogisticRegression(class_weight=weights)

In [14]:
model_bow_logreg = bow_logreg.fit(X_vec, y_rec)

### Random Forest Classifier

In [15]:
bow_rfc = RandomForestClassifier(
          n_estimators=100,
          class_weight=weights,
          random_state=16,
          n_jobs=-1
)

In [16]:
model_bow_rfc = bow_rfc.fit(X_vec, y_rec)

### Linear Regression

In [17]:
bow_linreg = LinearRegression()

In [18]:
model_bow_linreg = bow_linreg.fit(X_vec, y_rat)

### Random Forest Regressor

In [19]:
bow_rfr = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=16,
    n_jobs=-1
)

In [20]:
model_bow_rfr = bow_rfr.fit(X_vec, y_rat)

In [21]:
# joblib.dump(model_bow_logreg, 'bow_logreg_review_rec.joblib')
# joblib.dump(model_bow_rfc, 'bow_rfc_review_rec.joblib')
# joblib.dump(model_bow_linreg, 'bow_linreg_review_rat.joblib')
# joblib.dump(model_bow_rfr, 'bow_rfr_review_rat.joblib')

['bow_rfr_review_rat.joblib']

In [22]:
# joblib.dump(tfidf, 'vectorization_bow.joblib')

['vectorization_bow.joblib']

# Submission

* `model_bow_logreg` - для класифікації рекомендації за відгуком  
* `model_bow_rfr` - для визначення рейтингу за відгуком  
* `model_nr_rec` - для класифікації рекомендації БЕЗ відгука  
* `model_nr_rat` - для визначення рейтингу БЕЗ відгука  

In [23]:
model_nr_rec = joblib.load('models/no_review/model_nr_rec.joblib')
model_nr_rat = joblib.load('models/no_review/model_nr_rat.joblib')

In [24]:
test_bow_review = pd.read_csv('train_test_val/test/test_bow_review.csv')
test_rat_enc = pd.read_csv('train_test_val/test/test_rat_enc.csv')
test_rec_enc = pd.read_csv('train_test_val/test/test_rec_enc.csv')

In [25]:
test_bow_review.head()

,Id,clean_text
0,21403,magnificent clothe contrast reviewer love clot...
1,22553,shapeless tent try store huge not try small si...
2,17436,versatile think fun piece not realize versatil...
3,4293,simple cute buy multicolor stripe adorable ...
4,20149,magnificent simple tank wide strap style flatt...


In [26]:
test_rat_enc.head()

,Age,Pos_Feedback_Cnt,Division,Department,Product_Category
0,0.813273,0.277647,4.176844,4.145440,4.145440
1,0.649031,-0.084574,4.176844,4.145440,4.145440
2,1.305999,-0.265685,4.176844,4.318103,4.294331
3,0.402667,-0.265685,4.176844,4.145440,4.145440
4,0.238425,-0.446795,4.308628,4.308951,4.369983


In [27]:
test_rec_enc.head()

,Age,Pos_Feedback_Cnt,Division,Department,Product_Category
0,0.813273,0.277647,0.813338,0.806434,0.806434
1,0.649031,-0.084574,0.813338,0.806434,0.806434
2,1.305999,-0.265685,0.813338,0.852045,0.831122
3,0.402667,-0.265685,0.813338,0.806434,0.806434
4,0.238425,-0.446795,0.853982,0.855630,0.888740


In [51]:
idx_isna = test_bow_review['clean_text'].isna()

In [52]:
test_bow_review.shape, test_rat_enc.shape, test_rec_enc.shape

((9395, 2), (9395, 5), (9395, 5))

In [53]:
y_rec = pd.Series(index=test_bow_review.index, dtype=float)
y_rat = pd.Series(index=test_bow_review.index, dtype=float)

In [54]:
X_rev = tfidf.transform(test_bow_review.loc[~idx_isna, 'clean_text'])

In [55]:
X_nr_rec = test_rec_enc.loc[idx_isna]
X_nr_rat = test_rat_enc.loc[idx_isna]

In [56]:
y_rec[~idx_isna] = model_bow_logreg.predict(X_rev)
y_rat[~idx_isna] = model_bow_rfr.predict(X_rev)

In [57]:
y_rec[idx_isna] = model_nr_rec.predict(X_nr_rec)
y_rat[idx_isna] = model_nr_rat.predict(X_nr_rat)

In [58]:
y_rec = [ 1 if i > 0.5 
          else 0 for i in y_rec]

In [59]:
y_rat = [5 if i >= 4.5 
         else 4 if 3.5 <= i < 4.5 
         else 3 if 2.5 <= i < 3.5 
         else 2 if 1.5 <= i < 2.5 
         else 1 
         for i in y_rat]

In [60]:
sub_ds = pd.read_csv('final-project-danit-ds-3-4/sample_submission.csv')

In [61]:
sub_ds.head()

,Id,Rating,Recommended
0,21403,2,1
1,22553,5,0
2,17436,1,1
3,4293,1,0
4,20149,4,0


In [62]:
sub_ds['Rating'] = y_rat
sub_ds['Recommended'] = y_rec

In [63]:
sub_ds.head()

,Id,Rating,Recommended
0,21403,5,1
1,22553,3,0
2,17436,5,1
3,4293,5,1
4,20149,4,1


In [ ]:
sub_ds.to_csv('submission.csv', index=False)